# 1. icaa_id, distribuidoras, subvenciones, dirección y guión

Recibe `icaa_raw` del notebook 0 (extracción de PDFs) y hace todo lo que necesita
tocar el catálogo ICAA:

1. Resolver `icaa_id` por título (deduplicado, propagado)
2. Scrapear la ficha de cada `icaa_id` único: distribuidora real, género/tipo/tags,
   nacionalidad, **subvenciones** (concepto + importe, puede haber varias),
   **director(es)** y **guionista(s)** (puede haber varios de cada uno)
3. Agrupar en `icaa_peliculas` con la distribuidora de ficha como fuente principal
4. Guardar en MySQL + CSV

Peticiones con reintento y backoff exponencial ante 429 (Too Many Requests) --
confirmado que el sitio responde a `requests` normal, solo hay que ir despacio.

## Librerías y credenciales

In [1]:
import re
import time
import random
import json
import csv
from pathlib import Path

import requests
import pandas as pd
from bs4 import BeautifulSoup
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

from selenium import webdriver
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

load_dotenv()
BASE = Path("..")
print("✓ Imports OK")


✓ Imports OK


In [2]:
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
print("✓ engine creado")


✓ engine creado


## Cargar `icaa_raw` del notebook 0

In [44]:
icaa_raw = pd.read_csv(BASE / "3 - csv" / "icaa_raw_pdfs.csv", sep=';')
print(f"icaa_raw: {len(icaa_raw)} filas")
icaa_raw.head()


icaa_raw: 2812 filas


,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio,anio_reposicion,titulo_busqueda,articulo
0,A todo tren. Destino Asturias,Warner Bros,08/07/2021,8493358.0,1500811,2021,NaN,A todo tren. Destino Asturias,NaN
1,Way Down,Sony,12/11/2021,5628247.0,887897,2021,NaN,Way Down,NaN
2,Operacion Camaron,Walt Disney,24/06/2021,3522415.0,597700,2021,NaN,Operacion Camaron,NaN
3,"Buen patron, El",Tri Pictures,15/10/2021,3336892.0,528523,2021,NaN,Buen patron,El
4,Maixabel,Walt Disney,24/09/2021,2828416.0,515293,2021,NaN,Maixabel,NaN


## Selenium (Firefox) con reintentos ante 429

`requests` falla con `SSLCertVerificationError` incluso tras actualizar `certifi`
-- lo más probable es que `sede.mcu.gob.es` no mande la cadena de certificados
completa (falta el intermedio), algo típico en sedes de la administración
española. Los navegadores lo resuelven solos; el `ssl` de Python por defecto no.
Por eso volvemos a Selenium+Firefox (como ya tenías probado y estable), con un
único driver reutilizado por fase de scraping.

Los parsers (`parsear_resultados_busqueda`, `parsear_ficha_icaa`, etc.) NO cambian
-- siguen trabajando sobre un objeto `BeautifulSoup`, solo cambia de dónde sale el
HTML (`driver.page_source` en vez de `response.text`).

In [4]:
def iniciar_driver():
    """Un único driver de Firefox, reutilizable para toda una fase de scraping."""
    options = FirefoxOptions()
    options.add_argument("--headless")
    driver = webdriver.Firefox(options=options)
    driver.set_page_load_timeout(20)
    return driver


def _get_con_reintentos_selenium(driver, url, max_reintentos=6, espera_base=30, timeout=20):
    """
    Navega a `url` y devuelve el HTML. Reintenta con backoff exponencial si la
    respuesta es un 429 (Too Many Requests) -- Selenium no expone el status code
    HTTP directamente, así que se detecta por el contenido de la página (mismo
    mensaje de Apache que vimos con requests: "429 Too Many Requests").
    """
    for intento in range(max_reintentos):
        driver.get(url)
        try:
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
        except TimeoutException:
            pass

        if "429 Too Many Requests" in driver.page_source:
            espera = espera_base * (2 ** intento)
            print(f"  429 Too Many Requests -- esperando {espera:.0f}s (intento {intento + 1}/{max_reintentos})")
            time.sleep(espera)
            continue

        return driver.page_source

    raise RuntimeError(f"Demasiados 429 consecutivos para {url}")


print("✓ iniciar_driver y _get_con_reintentos_selenium definidas")


✓ iniciar_driver y _get_con_reintentos_selenium definidas


## Búsqueda avanzada por título

In [5]:
import urllib.parse

BASE_SEARCH_URL = "https://sede.mcu.gob.es/CatalogoICAA/es-es"
FICHA_URL = "https://sede.mcu.gob.es/CatalogoICAA/es-es/Peliculas/Detalle"


def construir_url_busqueda(titulo, pagina=None, ano_calificacion=None):
    params = {
        "sortOrder": "", "T_General": "", "Metraje": "", "Calificacion": "",
        "Ano": "", "Ano_Calificacion": str(ano_calificacion) if ano_calificacion else "",
        "Paiscopro": "", "Pais": "",
        "Titulo": titulo, "Director": "", "productora": "", "distribuidora": "",
        "Interprete": "", "Fotografia": "", "Guionista": "", "Musica": "",
        "gridlist": "list", "SoloEspana": "0", "SoloVideo": "0", "SoloPorno": "0",
    }
    if pagina and pagina > 1:
        params["page"] = pagina
    query = urllib.parse.urlencode(params, quote_via=urllib.parse.quote)
    return f"{BASE_SEARCH_URL}?{query}"


def _clean(texto):
    return re.sub(r"\s+", " ", texto or "").strip()


def parsear_resultados_busqueda(html):
    soup = BeautifulSoup(html, "html.parser")

    total_label = soup.find("span", class_="number-big")
    total_resultados = None
    total_truncado = False
    if total_label:
        texto_total = _clean(total_label.get_text())
        if texto_total.strip().startswith("+"):
            # ICAA no da conteo exacto por encima de cierto umbral (ej. "+ 1000")
            total_truncado = True
        else:
            m_total = re.search(r"\d+", texto_total)
            if m_total:
                total_resultados = int(m_total.group())

    resultados = []
    lista = soup.find("div", id="list-view")
    if not lista:
        return total_resultados, resultados, total_truncado

    for item in lista.find_all("div", class_="sin-list-pro"):
        titulo_a = item.find("a", class_="list-pro-title")
        if not titulo_a:
            continue

        href = titulo_a.get("href", "")
        m = re.search(r"Pelicula=(\d+)", href)
        icaa_id = int(m.group(1)) if m else None
        titulo = _clean(titulo_a.get_text())

        director_tag = (
            item.find("a", class_="custom-director-string")
            or item.find("p", class_="custom-director-string")
        )
        director = _clean(director_tag.get_text()) if director_tag else None

        anio_p = item.find("p", class_="list-pro-title mcu-result-text")
        anio = None
        if anio_p:
            m2 = re.search(r"\d{4}", anio_p.get_text())
            anio = int(m2.group()) if m2 else None

        metraje_label = item.find("label", class_="list-pro-title mcu-result-text")
        metraje = _clean(metraje_label.get_text()) if metraje_label else None

        resultados.append({
            "icaa_id": icaa_id,
            "titulo_encontrado": titulo,
            "director_encontrado": director,
            "anio_encontrado": anio,
            "metraje_encontrado": metraje,
        })

    return total_resultados, resultados, total_truncado


def buscar_titulo_icaa(titulo, driver, max_paginas=5, delay=(2.0, 4.0), ano_calificacion=None):
    todos = []
    total = None
    truncado_alguna_vez = False

    for pagina in range(1, max_paginas + 1):
        url = construir_url_busqueda(titulo, pagina=pagina if pagina > 1 else None, ano_calificacion=ano_calificacion)
        try:
            html = _get_con_reintentos_selenium(driver, url)
        except Exception as e:
            print(f"  Error en '{titulo}' (página {pagina}): {e}")
            break

        total, candidatos, truncado = parsear_resultados_busqueda(html)
        truncado_alguna_vez = truncado_alguna_vez or truncado
        if pagina == 1 and not candidatos:
            break
        todos.extend(candidatos)

        if truncado:
            if not candidatos:
                break
        elif total is None or len(todos) >= total or not candidatos:
            break
        time.sleep(random.uniform(*delay))

    if truncado_alguna_vez:
        print(f"  ⚠ '{titulo}': +1000 resultados totales -- término de búsqueda posiblemente demasiado genérico, revisar si no resuelve")

    return total, todos, truncado_alguna_vez


def _variantes_sin_apostrofe(titulo):
    """
    ICAA busca por subcadena LITERAL en servidor. Confirmado con datos reales:
    el catálogo NO omite el apóstrofo, lo SUSTITUYE por el carácter ´ (acento
    agudo, U+00B4) -- ej. "Guie'dani" en el PDF aparece como "GUIE´DANI" en la
    ficha. Por eso la variante prioritaria es sustituir por ´, no solo quitar
    el apóstrofo (que no basta si el catálogo conserva un carácter ahí).
    """
    variantes = []
    con_acento = re.sub(r"['\u2019`]", "´", titulo)  # el carácter real que usa ICAA
    sin_caracter = re.sub(r"['\u2019´`]", "", titulo)
    con_espacio = re.sub(r"['\u2019´`]", " ", titulo)
    con_espacio = re.sub(r"\s+", " ", con_espacio).strip()
    for v in (con_acento, sin_caracter, con_espacio):
        if v != titulo and v not in variantes:
            variantes.append(v)
    return variantes


def buscar_titulo_icaa_con_fallback_anio(titulo, anio_pdf, driver, max_paginas=5, delay=(2.0, 4.0)):
    """
    Si la búsqueda base sale truncada (+1000), reintenta filtrando por
    Ano_Calificacion en un rango de +/-1 año sobre el año de fecha_estreno del
    PDF (la calificación puede caer en diciembre/enero del año colindante).
    Confirmado manualmente: Ano_Calificacion sí filtra en servidor (de +1000 a
    14 resultados para "Noche" + 2021).

    Si la búsqueda no encuentra NINGÚN candidato y el título lleva apóstrofo,
    reintenta con variantes sin apóstrofo antes de rendirse.
    """
    total, candidatos, truncada = buscar_titulo_icaa(titulo, driver, max_paginas=max_paginas, delay=delay)

    if not candidatos and re.search(r"['\u2019´`]", titulo):
        for variante in _variantes_sin_apostrofe(titulo):
            print(f"  '{titulo}' sin candidatos -- reintentando sin apóstrofo: '{variante}'")
            _, cand_variante, trunc_variante = buscar_titulo_icaa(variante, driver, max_paginas=max_paginas, delay=delay)
            if cand_variante:
                candidatos = cand_variante
                truncada = trunc_variante
                break
            time.sleep(random.uniform(*delay))

    if not truncada or anio_pdf is None:
        return total, candidatos, truncada

    print(f"  '{titulo}' truncada -- reintentando con Ano_Calificacion cerca de {anio_pdf}")
    candidatos_por_id = {}
    for anio in (anio_pdf - 1, anio_pdf, anio_pdf + 1):
        _, cand_anio, _ = buscar_titulo_icaa(
            titulo, driver, max_paginas=max_paginas, delay=delay, ano_calificacion=anio
        )
        for c in cand_anio:
            candidatos_por_id[c["icaa_id"]] = c
        time.sleep(random.uniform(*delay))

    candidatos_combinados = list(candidatos_por_id.values())
    return total, (candidatos_combinados or candidatos), truncada


print("✓ funciones de búsqueda definidas (con fallback de año para búsquedas truncadas)")


✓ funciones de búsqueda definidas (con fallback de año para búsquedas truncadas)


## Parser de ficha de detalle

Todo lo que se saca en una sola petición por `icaa_id`: distribuidoras, tipo/género/tags,
nacionalidad y %, fecha de estreno real, **subvenciones**, **director(es)** y
**guionista(s)**.

In [6]:
def obtener_ficha(icaa_id, driver):
    url = f"{FICHA_URL}?Pelicula={icaa_id}"
    html = _get_con_reintentos_selenium(driver, url)
    return BeautifulSoup(html, "html.parser")


def extraer_fecha_estreno_ficha(soup):
    label = soup.find(
        "label", class_="mcu-text-details-text-b", string=re.compile("Fecha de Estreno")
    )
    if not label:
        return None
    valor = label.find_next_sibling("label", class_="custom-simple-label")
    return _clean(valor.get_text()) if valor else None


def parsear_distribuidoras(soup):
    """
    OJO: la ficha ICAA repite id="p_empresas" en DOS divs distintos (Empresas
    Productoras y Datos y Empresas Distribuidoras) -- id duplicado inválido del
    propio ICAA. No se puede anclar por id; se busca el <li> de cabecera correcto.
    """
    resultado = {"distribuidora_nacional_icaa": None, "distribuidor_internacional_icaa": None}

    li_distribucion = soup.find("li", string=re.compile(r"DATOS Y EMPRESAS DISTRIBUIDORAS"))
    empresas_div = li_distribucion.find_parent("div", id="p_empresas") if li_distribucion else None
    if not empresas_div:
        return resultado

    for bloque in empresas_div.find_all("div", class_="header-panel-details"):
        etiqueta = bloque.find("label", style=lambda s: s and "font-weight:500" in s)
        if not etiqueta:
            continue
        nombre_seccion = _clean(etiqueta.get_text())

        contenido = bloque.find("div", class_="hidden-panel-details")
        empresas = []
        if contenido:
            for label_empresa in contenido.select(
                ".header-panel-details-child > .custom-header-text > label"
            ):
                nombre = _clean(label_empresa.get_text())
                if nombre:
                    empresas.append(nombre)

        valor = "; ".join(empresas) if empresas else None
        if nombre_seccion == "Distribuidora Nacional":
            resultado["distribuidora_nacional_icaa"] = valor
        elif nombre_seccion == "Distribuidor Internacional - Ventas":
            resultado["distribuidor_internacional_icaa"] = valor

    return resultado


def parsear_tipo_genero_tags(soup):
    resultado = {"tipo_icaa": None, "genero_icaa": None, "tags_icaa": None}

    tipo_label = soup.find("label", class_="mcu-text-details-text-b", string=re.compile(r"Tipo:"))
    if tipo_label:
        valor_tipo = tipo_label.find_next_sibling("label", class_="custom-simple-label")
        if valor_tipo:
            resultado["tipo_icaa"] = _clean(valor_tipo.get_text())
            genero_label = valor_tipo.find_next_sibling("label", class_="mcu-text-details-text-b")
            if genero_label and "nero" in genero_label.get_text():
                genero_valor = genero_label.find_next_sibling("label", class_="custom-simple-label")
                if genero_valor:
                    resultado["genero_icaa"] = _clean(genero_valor.get_text())

    tags = soup.find_all("label", class_="mcu-tags")
    if tags:
        resultado["tags_icaa"] = [_clean(t.get_text()) for t in tags]

    return resultado


def resolver_genero_final(tipo_icaa, genero_icaa, tags_icaa):
    if genero_icaa:
        return genero_icaa, "ficha_genero"
    if tipo_icaa:
        return tipo_icaa, "ficha_tipo"
    if tags_icaa:
        return tags_icaa[0], "tag"
    return None, None


def parsear_nacionalidad_porcentajes(soup):
    nac_label = soup.find("label", class_="mcu-text-details-text-b", string=re.compile("Nacionalidad"))
    if not nac_label:
        return []
    contenedor = nac_label.find_parent("div", class_="product-details-content")
    if not contenedor:
        return []

    porcentaje_label = contenedor.find("label", style=lambda s: s and "font-size: 11px" in s)
    if porcentaje_label:
        texto = _clean(porcentaje_label.get_text())
        pares = re.findall(r"([A-Za-zÀ-ÿ.\s]+?)\s*\(([\d.]+)%\)", texto)
        return [(pais.strip(), float(pct)) for pais, pct in pares]

    pais_unico = contenedor.find("label", class_="custom-simple-label")
    if pais_unico:
        nombre = _clean(pais_unico.get_text())
        if nombre:
            return [(nombre, 100.0)]
    return []


def parsear_direccion_guion(soup):
    """
    Extrae directores y guionistas desde la tabla FICHA TÉCNICA (id="p_tecnicos").
    Puede haber varios de cada uno (codirección, guión coral). Algunas fichas no
    traen tabla técnica en absoluto (ej. icaa_id=9221) -> listas vacías.
    """
    resultado = {"directores_icaa": [], "guionistas_icaa": []}
    tabla = soup.find("div", id="p_tecnicos")
    if not tabla:
        return resultado
    cuerpo = tabla.find("tbody")
    if not cuerpo:
        return resultado
    for fila in cuerpo.find_all("tr"):
        celdas = fila.find_all("td")
        if len(celdas) < 2:
            continue
        funcion = _clean(celdas[0].get_text())
        nombre = _clean(celdas[1].get_text())
        if not nombre:
            continue
        if funcion == "Dirigido por":
            resultado["directores_icaa"].append(nombre)
        elif funcion == "Guión":
            resultado["guionistas_icaa"].append(nombre)
    return resultado


def _parsear_importe(texto):
    """'1.000.000,00 €' -> 1000000.00 -- validado contra 155 fichas reales (notebook 4)."""
    if not texto:
        return None
    limpio = re.sub(r"[^\d,]", "", texto)
    limpio = limpio.replace(".", "").replace(",", ".")
    try:
        return float(limpio)
    except ValueError:
        return None


def parsear_subvenciones(soup):
    """
    Subvenciones públicas, ancladas en id="p_ayudas" (arreglo del bug original:
    buscar por texto "SUBVENCION" hacía match con un div envolvente de toda la
    página). Puede haber varias filas por película. Validado contra 155 fichas
    reales: 74 filas de subvención en 72 películas.
    """
    subvenciones = []
    contenedor = soup.find(id="p_ayudas")
    if not contenedor:
        return subvenciones
    tabla = contenedor.find("table")
    if not tabla:
        return subvenciones
    for fila in tabla.find_all("tr"):
        celdas = fila.find_all(["td", "th"])
        if len(celdas) < 2:
            continue
        concepto = _clean(celdas[0].get_text())
        importe = _parsear_importe(celdas[1].get_text())
        if concepto and importe is not None:
            subvenciones.append({"concepto": concepto, "importe": importe})
    return subvenciones


def parsear_empresas_productoras(soup):
    """
    Extrae las empresas productoras de la película, agrupadas por país de
    coproducción, con el porcentaje de participación de cada empresa y el
    porcentaje total del país. Igual que en distribuidoras, la ficha ICAA
    repite id="p_empresas" en dos divs distintos (productoras y
    distribuidoras) -- se ancla por el <li> de cabecera correcto.
    """
    resultado = []
    li_productoras = soup.find("li", string=re.compile(r"EMPRESAS PRODUCTORAS"))
    empresas_div = li_productoras.find_parent("div", id="p_empresas") if li_productoras else None
    if not empresas_div:
        return resultado

    for bloque_pais in empresas_div.find_all("div", class_="header-panel-details"):
        etiqueta_pais = bloque_pais.find("label")
        if not etiqueta_pais:
            continue
        texto_pais = _clean(etiqueta_pais.get_text())
        m_pais = re.match(r"^(.+?)\s*\(?([\d.]+)%\)?$", texto_pais)
        pais, porcentaje_pais = (m_pais.group(1).strip(), float(m_pais.group(2))) if m_pais else (texto_pais, None)

        contenido = bloque_pais.find("div", class_="hidden-panel-details")
        if not contenido:
            continue
        for label_empresa in contenido.select(".header-panel-details-child > .custom-header-text > label"):
            texto_empresa = _clean(label_empresa.get_text())
            m_emp = re.match(r"^(.+?)\s*([\d.]+)%$", texto_empresa)
            empresa, porcentaje_empresa = (m_emp.group(1).strip(), float(m_emp.group(2))) if m_emp else (texto_empresa, None)
            resultado.append({
                "pais": pais, "porcentaje_pais": porcentaje_pais,
                "empresa": empresa, "porcentaje_empresa": porcentaje_empresa,
            })
    return resultado


def extraer_titulo_ficha(soup):
    """
    Título oficial del catálogo tal como aparece en la ficha (h2.custom-detail-title):
    limpio, sin sufijos de reposición ni artefactos del PDF, y con el artículo en
    la forma en que ICAA lo tenga (antepuesto, pospuesto, o sin él).
    """
    h2 = soup.find("h2", class_="custom-detail-title")
    return _clean(h2.get_text()) if h2 else None


def parsear_ficha_icaa(soup, icaa_id):
    datos = {"icaa_id": icaa_id}
    datos["titulo_icaa"] = extraer_titulo_ficha(soup)
    datos.update(parsear_distribuidoras(soup))
    datos["fecha_estreno_ficha"] = extraer_fecha_estreno_ficha(soup)

    tgt = parsear_tipo_genero_tags(soup)
    datos.update(tgt)
    genero_final, genero_fuente = resolver_genero_final(
        tgt["tipo_icaa"], tgt["genero_icaa"], tgt["tags_icaa"]
    )
    datos["genero_final"] = genero_final
    datos["genero_fuente"] = genero_fuente

    nacionalidad = parsear_nacionalidad_porcentajes(soup)
    datos["nacionalidad_paises_icaa"] = (
        "; ".join(f"{p} ({pct}%)" for p, pct in nacionalidad) if nacionalidad else None
    )

    dyg = parsear_direccion_guion(soup)
    datos["directores_icaa"] = dyg["directores_icaa"]
    datos["guionistas_icaa"] = dyg["guionistas_icaa"]

    subvenciones = parsear_subvenciones(soup)
    datos["subvenciones_icaa"] = subvenciones
    datos["subvenciones_total"] = sum(s["importe"] for s in subvenciones) if subvenciones else 0.0

    datos["empresas_productoras_icaa"] = parsear_empresas_productoras(soup)

    return datos


print("✓ parser de ficha definido (con subvenciones, director y guión)")


✓ parser de ficha definido (con subvenciones, director y guión)


## Desambiguación de candidatos

**Normalización mejorada**: además de tildes, ahora también se quita toda la
puntuación general (comas que no son de artículo, puntos, ¡!¿?, comillas,
paréntesis...) al comparar títulos. Esto resuelve casos como:
- `"¿Capaz o Incapaz?"` vs catálogo `"Capaz o incapaz?"` (¿ de más/menos)
- `"10,000 KM"` vs catálogo `"10.000 KM"` (separador de miles distinto)
- `"¡Dolores, guapa!"` vs catálogo `"¡Dolores Guapa!"` (coma intermedia que no es de artículo)

La coma del artículo pospuesto (`"Título, El"`) ya se gestiona aparte en
`separar_articulo` (notebook 0) antes de llegar aquí, así que quitar el resto
de la puntuación en la comparación no pierde esa información.

In [7]:
import unicodedata

def _quitar_tildes(texto):
    return ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))


def _normaliza_titulo(t):
    """
    Normalización para comparar títulos: minúsculas, sin tildes, sin puntuación
    general (comas, puntos, ¡!¿?, comillas, paréntesis, dos puntos...), espacios
    colapsados. No toca la separación de artículo -- eso ya viene resuelto desde
    notebook 0 (separar_articulo) antes de que el título llegue aquí.
    """
    t = (t or "").lower()
    t = _quitar_tildes(t)
    t = re.sub(r"[^\w\s]", " ", t, flags=re.UNICODE)
    t = re.sub(r"\s+", " ", t).strip()
    return t


def formas_titulo(titulo, titulo_busqueda, articulo):
    """
    Genera las formas válidas de título para verificar coincidencia exacta,
    dado que ICAA no es consistente en dónde coloca el artículo (a veces
    pospuesto como el PDF, a veces antepuesto, a veces sin él). titulo_busqueda
    y articulo vienen ya calculados desde el notebook 0.

    OJO: 'titulo' es el original SIN limpiar (puede traer sufijos de reposición
    tipo "(1985) (re)"), así que no sirve por sí solo como forma pospuesta
    limpia cuando hay reposición -- hay que reconstruirla explícitamente a
    partir de titulo_busqueda + articulo.
    """
    formas = {titulo, titulo_busqueda}
    if isinstance(articulo, str) and articulo:
        formas.add(f"{articulo} {titulo_busqueda}")       # antepuesto: "La Vaquilla"
        formas.add(f"{titulo_busqueda}, {articulo}")       # pospuesto limpio: "Vaquilla, La"
    return formas


def extraer_anio(fecha):
    if fecha is None or (isinstance(fecha, float) and pd.isna(fecha)):
        return None
    m = re.search(r"(\d{4})", str(fecha))
    return int(m.group(1)) if m else None


def elegir_mejor_candidato(formas_titulo_pdf, fecha_estreno_pdf, anio_reposicion, candidatos, driver, delay=(2.0, 4.0)):
    """
    1. Título exacto (normalizado, contra CUALQUIERA de las formas del título
       -- con artículo pospuesto, antepuesto, o sin artículo).
       - 0 -> sin_coincidencia_exacta_titulo (probable título en otro idioma en catálogo)
       - 1 -> resuelto directo
       - 2+ -> paso 2
    2. Filtrar por |Año de Producción - año_referencia| < 5, probando dos años de
       referencia: el año de fecha_estreno del PDF, Y (si existe) anio_reposicion
       -- el año capturado de anotaciones tipo "(1985) (re)", que suele ser el año
       de producción real. Para reposiciones, fecha_estreno es la fecha de la
       REPOSICIÓN (puede ser décadas después del año de producción), así que
       depender solo de ese año descartaba casi todas las reposiciones.
       - 0 -> sin_candidatos_cercanos (sin match ni por fecha_estreno ni por año de reposición)
       - 1 -> resuelto
       - 2+ -> paso 3
    3. Fallback a ficha real (solo para los candidatos que sobrevivieron el filtro
       de 5 años):
       a. Fecha de Estreno EXACTA coincide con fecha_estreno_pdf -> resuelto
       b. si no, año de esa Fecha de Estreno coincide con año(fecha_estreno_pdf) -> resuelto
       c. si no -> ambigua, revisión manual
    """
    anio_pdf = extraer_anio(fecha_estreno_pdf)
    formas_norm = {_normaliza_titulo(f) for f in formas_titulo_pdf}
    exactos = [c for c in candidatos if _normaliza_titulo(c["titulo_encontrado"]) in formas_norm]

    if not exactos:
        return None, "sin_coincidencia_exacta_titulo"

    if len(exactos) == 1:
        return exactos[0], "resuelto_titulo_unico"

    anios_referencia = [a for a in (anio_pdf, anio_reposicion) if a is not None]
    if not anios_referencia:
        return None, "ambigua_sin_anio_pdf"

    cercanos = [
        c for c in exactos
        if c["anio_encontrado"] is not None
        and any(abs(c["anio_encontrado"] - a) < 5 for a in anios_referencia)
    ]

    if len(cercanos) == 0:
        return None, "sin_candidatos_cercanos_probable_reposicion"
    if len(cercanos) == 1:
        return cercanos[0], "resuelto_anio_produccion_cercano"

    fichas = []
    for c in cercanos:
        try:
            soup = obtener_ficha(c["icaa_id"], driver)
            fecha_ficha = extraer_fecha_estreno_ficha(soup)
            time.sleep(random.uniform(*delay))
        except Exception as e:
            print(f"  Error consultando ficha {c['icaa_id']}: {e}")
            continue
        c2 = dict(c)
        c2["fecha_estreno_ficha"] = fecha_ficha
        fichas.append(c2)

    exactas_fecha = [c for c in fichas if c["fecha_estreno_ficha"] == fecha_estreno_pdf]
    if len(exactas_fecha) == 1:
        return exactas_fecha[0], "resuelto_fecha_estreno_exacta"

    exactas_anio = [c for c in fichas if extraer_anio(c["fecha_estreno_ficha"]) == anio_pdf]
    if len(exactas_anio) == 1:
        return exactas_anio[0], "resuelto_anio_estreno_ficha"

    return None, "ambigua_tras_ficha"


print("✓ elegir_mejor_candidato definida (normalización con puntuación general)")


✓ elegir_mejor_candidato definida (normalización con puntuación general)


## Resolución de `icaa_id`: deduplicar (titulo, fecha_estreno) y buscar una sola vez

### Ruta de la caché de resolución de `icaa_id`

Se define aquí, antes de los overrides y del auto-completado por tildes, porque
ambas celdas necesitan poder leer/escribir esta caché sin depender de que ya se
haya calculado `claves_pendientes` más abajo.

In [8]:
CACHE_ICAA_ID = BASE / "3 - csv" / "icaa_id_busqueda_cache.csv"
CAMPOS_CACHE_ID = [
    "titulo_original", "fecha_estreno_original", "titulo_busqueda_usado", "icaa_id", "estado",
    "titulo_encontrado", "anio_encontrado", "fecha_estreno_ficha",
    "director_encontrado", "num_candidatos", "candidatos_json", "busqueda_truncada",
]
print(f"✓ CACHE_ICAA_ID = {CACHE_ICAA_ID}")


✓ CACHE_ICAA_ID = ..\3 - csv\icaa_id_busqueda_cache.csv


In [9]:
CACHE_FICHAS = BASE / "3 - csv" / "ficha_icaa_cache.csv"
CAMPOS_FICHA = [
    "icaa_id", "titulo_icaa", "distribuidora_nacional_icaa", "distribuidor_internacional_icaa",
    "fecha_estreno_ficha", "tipo_icaa", "genero_icaa", "tags_icaa",
    "genero_final", "genero_fuente", "nacionalidad_paises_icaa",
    "directores_icaa", "guionistas_icaa",
    "subvenciones_icaa", "subvenciones_total",
    "empresas_productoras_icaa",
]
print(f"✓ CACHE_FICHAS = {CACHE_FICHAS}")


✓ CACHE_FICHAS = ..\3 - csv\ficha_icaa_cache.csv


In [13]:
OVERRIDES_ICAA_ID = {
    # ('TITULO EXACTO EN icaa_raw', '17/11/2023'): 155822,
}
print(f"✓ {len(OVERRIDES_ICAA_ID)} overrides de icaa_id cargados")


✓ 0 overrides de icaa_id cargados


### Auto-completado de overrides por tildes/puntuación

Para los ya marcados `sin_coincidencia_exacta_titulo`/`ambigua_*`: revisa si,
con la normalización completa (tildes + puntuación general), el título coincide
con **una única** película entre los candidatos ya guardados en caché (sin
volver a golpear la red). Si hay más de una coincidencia, no decide por ti --
lo deja para revisión manual real. Borra de la caché las filas que resuelve,
para que vuelvan a quedar "pendientes" en la siguiente pasada del bucle.

In [10]:
import json as _json

cache_para_revisar = pd.read_csv(CACHE_ICAA_ID, sep=';')
pendientes_revision = cache_para_revisar[
    cache_para_revisar['icaa_id'].isna() & cache_para_revisar['candidatos_json'].notna()
]

claves_resueltas = []
for _, fila in pendientes_revision.iterrows():
    titulo = fila['titulo_original']
    fecha = fila['fecha_estreno_original']
    candidatos = _json.loads(fila['candidatos_json'])

    formas_norm = {_normaliza_titulo(titulo)}
    if pd.notna(fila.get('titulo_busqueda_usado')):
        formas_norm.add(_normaliza_titulo(fila['titulo_busqueda_usado']))

    coincidencias = [c for c in candidatos if _normaliza_titulo(c['titulo_encontrado']) in formas_norm]

    if len(coincidencias) == 1:
        clave = (titulo, fecha)
        OVERRIDES_ICAA_ID[clave] = coincidencias[0]['icaa_id']
        claves_resueltas.append(clave)
        print(f"✓ '{titulo}' -> icaa_id={coincidencias[0]['icaa_id']} (\"{coincidencias[0]['titulo_encontrado']}\")")
    elif len(coincidencias) > 1:
        print(f"⚠ '{titulo}': {len(coincidencias)} coincidencias tras quitar tildes -- revisar a mano")

print(f"\n{len(claves_resueltas)} overrides añadidos automáticamente a OVERRIDES_ICAA_ID")

if claves_resueltas:
    cache_para_revisar = cache_para_revisar[
        ~cache_para_revisar.apply(lambda f: (f['titulo_original'], f['fecha_estreno_original']) in claves_resueltas, axis=1)
    ]
    cache_para_revisar.to_csv(CACHE_ICAA_ID, index=False, sep=';')
    print(f"Filas eliminadas de la caché: {len(claves_resueltas)}. Filas restantes: {len(cache_para_revisar)}")


⚠ 'Domino': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Amanecer': 15 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Aun': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Elegido, El': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Cambio de sentido': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Casting': 5 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Vida empieza hoy, La': 3 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Dulcinea': 4 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Ultimo deseo, El': 3 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Madres': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Gran familia, La': 2 coincidencias tras quitar tildes -- revisar a mano
⚠ 'Epilogo': 3 coincidencias tras quitar tildes -- revisar a mano

0 overrides añadidos automáticamente a OVERRIDES_ICAA_ID


In [21]:
claves_unicas = icaa_raw[['titulo', 'titulo_busqueda', 'articulo', 'fecha_estreno', 'anio_reposicion']].drop_duplicates().reset_index(drop=True)
print(f"Combinaciones únicas (titulo, fecha_estreno): {len(claves_unicas)} / {len(icaa_raw)} filas totales")

ya_procesados = set()
if CACHE_ICAA_ID.exists():
    cache_previo = pd.read_csv(CACHE_ICAA_ID, sep=';')
    ya_procesados = set(zip(cache_previo['titulo_original'], cache_previo['fecha_estreno_original']))
    print(f"Reanudando: {len(ya_procesados)} claves ya procesadas")
else:
    print("Empezando desde cero")

claves_pendientes = claves_unicas[
    ~claves_unicas.apply(lambda f: (f['titulo'], f['fecha_estreno']) in ya_procesados, axis=1)
]
print(f"Pendientes: {len(claves_pendientes)} / {len(claves_unicas)}")


Combinaciones únicas (titulo, fecha_estreno): 2079 / 2812 filas totales
Reanudando: 2075 claves ya procesadas
Pendientes: 4 / 2079


In [22]:
driver = iniciar_driver()
write_header = not CACHE_ICAA_ID.exists()
fallos_consecutivos = 0
MAX_FALLOS_ANTES_DE_REINICIAR = 3

try:
    with open(CACHE_ICAA_ID, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS_CACHE_ID, delimiter=';', quoting=csv.QUOTE_NONNUMERIC)
        if write_header:
            writer.writeheader()

        for i, (_, fila) in enumerate(claves_pendientes.iterrows(), 1):
            titulo = fila['titulo']
            titulo_busqueda = fila['titulo_busqueda']
            articulo = fila['articulo']
            fecha_estreno_pdf = fila['fecha_estreno']
            clave = (titulo, fecha_estreno_pdf)

            if i % 50 == 0:
                print(f"[{i}/{len(claves_pendientes)}] {titulo[:50]}")

            if clave in OVERRIDES_ICAA_ID:
                fila_out = {
                    "titulo_original": titulo, "fecha_estreno_original": fecha_estreno_pdf,
                    "titulo_busqueda_usado": titulo_busqueda,
                    "icaa_id": OVERRIDES_ICAA_ID[clave], "estado": "override_manual",
                    "titulo_encontrado": None, "anio_encontrado": None, "fecha_estreno_ficha": None,
                    "director_encontrado": None, "num_candidatos": None, "candidatos_json": None,
                    "busqueda_truncada": False,
                }
                fallos_consecutivos = 0
            else:
                try:
                    anio_pdf = extraer_anio(fecha_estreno_pdf)
                    anio_reposicion = fila['anio_reposicion']
                    anio_reposicion = int(anio_reposicion) if pd.notna(anio_reposicion) else None

                    total, candidatos, truncada = buscar_titulo_icaa_con_fallback_anio(titulo_busqueda, anio_pdf, driver)
                    formas = formas_titulo(titulo, titulo_busqueda, articulo)
                    elegido, estado = elegir_mejor_candidato(formas, fecha_estreno_pdf, anio_reposicion, candidatos, driver)

                    fila_out = {
                        "titulo_original": titulo,
                        "fecha_estreno_original": fecha_estreno_pdf,
                        "titulo_busqueda_usado": titulo_busqueda,
                        "icaa_id": elegido["icaa_id"] if elegido else None,
                        "estado": estado,
                        "titulo_encontrado": elegido["titulo_encontrado"] if elegido else None,
                        "anio_encontrado": elegido["anio_encontrado"] if elegido else None,
                        "fecha_estreno_ficha": elegido.get("fecha_estreno_ficha") if elegido else None,
                        "director_encontrado": elegido["director_encontrado"] if elegido else None,
                        "num_candidatos": len(candidatos),
                        "candidatos_json": json.dumps(candidatos, ensure_ascii=False) if not elegido else None,
                        "busqueda_truncada": truncada,
                    }
                    fallos_consecutivos = 0
                except Exception as e:
                    print(f"  Error en '{titulo}': {e}")
                    fila_out = {
                        "titulo_original": titulo, "fecha_estreno_original": fecha_estreno_pdf,
                        "titulo_busqueda_usado": titulo_busqueda,
                        "icaa_id": None, "estado": "error_navegacion",
                        "titulo_encontrado": None, "anio_encontrado": None, "fecha_estreno_ficha": None,
                        "director_encontrado": None, "num_candidatos": None, "candidatos_json": None,
                        "busqueda_truncada": False,
                    }
                    fallos_consecutivos += 1
                    if fallos_consecutivos >= MAX_FALLOS_ANTES_DE_REINICIAR:
                        print(f"  ⚠ {fallos_consecutivos} fallos seguidos -- reiniciando el driver de Firefox")
                        try:
                            driver.quit()
                        except Exception:
                            pass
                        time.sleep(5)
                        driver = iniciar_driver()
                        fallos_consecutivos = 0

                time.sleep(random.uniform(2.0, 4.0))

            writer.writerow(fila_out)
            f.flush()
finally:
    driver.quit()
    print("✓ Driver cerrado")

print(f"\n✓ Completado -> {CACHE_ICAA_ID}")


✓ Driver cerrado

✓ Completado -> ..\3 - csv\icaa_id_busqueda_cache.csv


In [23]:
resultado_busqueda = pd.read_csv(CACHE_ICAA_ID, sep=';')
print(resultado_busqueda['estado'].value_counts())
con_id = resultado_busqueda['icaa_id'].notna().sum()
print(f"\nCon icaa_id resuelto: {con_id} / {len(resultado_busqueda)} ({con_id/len(resultado_busqueda)*100:.1f}%)")

# .astype(bool) sobre texto es una trampa: "False" (string) también da True.
# Comparamos explícitamente contra los valores textuales/booleanos posibles.
truncadas = resultado_busqueda['busqueda_truncada'].isin([True, 'True', 'TRUE']).sum()
print(f"Búsquedas truncadas (+1000 resultados, término posiblemente muy genérico): {truncadas}")
if truncadas > 0:
    mascara_truncadas = resultado_busqueda['busqueda_truncada'].isin([True, 'True', 'TRUE'])
    print(resultado_busqueda[mascara_truncadas][['titulo_original', 'titulo_busqueda_usado', 'icaa_id', 'estado']])


estado
resuelto_titulo_unico                          1252
override_manual                                 524
sin_coincidencia_exacta_titulo                  263
resuelto_anio_produccion_cercano                130
resuelto_anio_estreno_ficha                      14
resuelto_fecha_estreno_exacta                    13
sin_candidatos_cercanos_probable_reposicion      11
ambigua_tras_ficha                                3
Name: count, dtype: int64

Con icaa_id resuelto: 1933 / 2210 (87.5%)
Búsquedas truncadas (+1000 resultados, término posiblemente muy genérico): 1
    titulo_original titulo_busqueda_usado  icaa_id                 estado
265       Noche, La                 Noche  55020.0  resuelto_titulo_unico


### Listado de pendientes (con y sin candidatos)

**Arreglado**: antes, una fila con `candidatos_json = "[]"` (lista vacía pero
no nula) se contaba como "con candidatos" solo por no ser `NaN`. Ahora se
parsea el JSON y se cuenta el nº real de candidatos.

In [24]:
import json

resultado_busqueda = pd.read_csv(CACHE_ICAA_ID, sep=';')
pendientes = resultado_busqueda[resultado_busqueda['icaa_id'].isna()].copy()

def _tiene_candidatos(valor_json):
    if pd.isna(valor_json):
        return False
    try:
        return len(json.loads(valor_json)) > 0
    except Exception:
        return False

mask_con_candidatos = pendientes['candidatos_json'].apply(_tiene_candidatos)

print(f"Total pendientes: {len(pendientes)}\n")
print("--- Desglose por estado ---")
print(pendientes['estado'].value_counts())
print()

con_candidatos = pendientes[mask_con_candidatos]
sin_candidatos = pendientes[~mask_con_candidatos]
print(f"Con candidatos (para revisar y elegir): {len(con_candidatos)}")
print(f"Sin ningún candidato (probable título en otro idioma / no está en catálogo): {len(sin_candidatos)}")
print()

print("=" * 70)
print("SIN NINGÚN CANDIDATO")
print("=" * 70)
for _, fila in sin_candidatos.iterrows():
    print(f"  {fila['titulo_original']}  ({fila['fecha_estreno_original']})  [buscado: '{fila['titulo_busqueda_usado']}']")

print()
print("=" * 70)
print("CON CANDIDATOS (revisar y decidir)")
print("=" * 70)
for _, fila in con_candidatos.iterrows():
    print(f"\n── {fila['titulo_original']}  ({fila['fecha_estreno_original']})  [{fila['estado']}]")
    candidatos = json.loads(fila['candidatos_json'])
    for c in candidatos[:8]:
        print(f"     icaa_id={c['icaa_id']:<10} {c['titulo_encontrado']:<50} "
              f"({c['anio_encontrado']}) — {c['director_encontrado']}")
    if len(candidatos) > 8:
        print(f"     ... y {len(candidatos) - 8} más")


Total pendientes: 277

--- Desglose por estado ---
estado
sin_coincidencia_exacta_titulo                 263
sin_candidatos_cercanos_probable_reposicion     11
ambigua_tras_ficha                               3
Name: count, dtype: int64

Con candidatos (para revisar y elegir): 35
Sin ningún candidato (probable título en otro idioma / no está en catálogo): 242

SIN NINGÚN CANDIDATO
  Pica Pica Musical Especial Navidad  (18/12/2020)  [buscado: 'Pica Pica Musical Especial Navidad']
  Viva montesa. La vida de un sueño  (16/11/2021)  [buscado: 'Viva montesa. La vida de un sueño']
  Retorno, El: la vida despues del ISIS  (27/05/2021)  [buscado: 'Retorno, El: la vida despues del ISIS']
  Salvador Dali. Diarios de juventud  (09/03/2021)  [buscado: 'Salvador Dali. Diarios de juventud']
  Basotik itsasora (Del bosque al mar)  (05/06/2021)  [buscado: 'Basotik itsasora (Del bosque al mar)']
  De Dali a Miquel  (09/04/2021)  [buscado: 'De Dali a Miquel']
  Canciones de horror  (16/10/2021)  [buscad

In [25]:
import json

resultado_busqueda = pd.read_csv(CACHE_ICAA_ID, sep=';')
pendientes = resultado_busqueda[resultado_busqueda['icaa_id'].isna()].copy()

print(f"# {len(pendientes)} títulos pendientes -- rellena los icaa_id (o deja None si no aplica) y pega el bloque")
print("OVERRIDES_ICAA_ID_NUEVOS = {")
for _, fila in pendientes.iterrows():
    titulo = fila['titulo_original']
    fecha = fila['fecha_estreno_original']
    comentario = ""
    if pd.notna(fila['candidatos_json']):
        candidatos = json.loads(fila['candidatos_json'])
        if candidatos:
            top = candidatos[0]
            comentario = f"  # ej: {top['icaa_id']} = {top['titulo_encontrado']} ({top['anio_encontrado']})"
    print(f"    ({titulo!r}, {fecha!r}): None,{comentario}")
print("}")

# 277 títulos pendientes -- rellena los icaa_id (o deja None si no aplica) y pega el bloque
OVERRIDES_ICAA_ID_NUEVOS = {
    ('Pica Pica Musical Especial Navidad', '18/12/2020'): None,
    ('Viva montesa. La vida de un sueño', '16/11/2021'): None,
    ('Retorno, El: la vida despues del ISIS', '27/05/2021'): None,
    ('Salvador Dali. Diarios de juventud', '09/03/2021'): None,
    ('Basotik itsasora (Del bosque al mar)', '05/06/2021'): None,
    ('De Dali a Miquel', '09/04/2021'): None,
    ('Canciones de horror', '16/10/2021'): None,
    ('Cumbre es el camino, La', '24/03/2021'): None,
    ('Un tiempo precioso', '17/07/2020'): None,
    ('Pasion por el cine: trio musical', '09/09/2021'): None,
    ('Palabra maldita, La', '29/10/2021'): None,
    ('Ez, Eskerrik asko! La ventana de Gladys', '23/03/2021'): None,
    ('No hay piedad para los condenados', '10/12/2021'): None,
    ('Depedro - Todo va a salir bien', '16/12/2021'): None,
    ('Yo mate a Ralph Greene', '20/03/2021'): None,
    

In [ ]:
# 277 títulos pendientes -- rellena los icaa_id (o deja None si no aplica) y pega el bloque
# ---------------------------------- TODITO ESTO YA LO REVISÉ MANUALMENTE SIN RESULTADO ---------------------------------------------
OVERRIDES_ICAA_ID_NUEVOS = {
    ('Pica Pica Musical Especial Navidad', '18/12/2020'): None,
    ('Viva montesa. La vida de un sueño', '16/11/2021'): None,
    ('Retorno, El: la vida despues del ISIS', '27/05/2021'): None,
    ('Salvador Dali. Diarios de juventud', '09/03/2021'): None,
    ('Basotik itsasora (Del bosque al mar)', '05/06/2021'): None,
    ('De Dali a Miquel', '09/04/2021'): None,
    ('Canciones de horror', '16/10/2021'): None,
    ('Cumbre es el camino, La', '24/03/2021'): None,
    ('Un tiempo precioso', '17/07/2020'): None,
    ('Pasion por el cine: trio musical', '09/09/2021'): None,
    ('Palabra maldita, La', '29/10/2021'): None,
    ('Ez, Eskerrik asko! La ventana de Gladys', '23/03/2021'): None,
    ('No hay piedad para los condenados', '10/12/2021'): None,
    ('Depedro - Todo va a salir bien', '16/12/2021'): None,
    ('Yo mate a Ralph Greene', '20/03/2021'): None,
    ('Piedra patria', '12/12/2021'): None,
    ('Nicaragua, patria libre para vivir', '29/04/2021'): None,
    ('Bury Us! A Punk Rock Uprising', '05/06/2021'): None,
    ('Aguantando er tipo', '30/09/2021'): None,
    ('Jantzari: tradicion e igualdad', '28/04/2021'): None,
    ('Viaje de Carla, El', '28/06/2021'): None,
    ('Ojala mañana', '01/12/2021'): None,
    ('Palabra justa, La', '19/02/2018'): None,
    ('Cosas que hacer antes de morir', '03/06/2021'): None,
    ('Sopar, El (1974-2018)', '15/04/2021'): None,
    ('Mujeres de arena', '09/04/2021'): None,
    ('Domino', '21/02/2020'): None,  # ej: 601130 = DAMA DEL DOMINO VERDE, LA (1937)
    ('Figuras femeninas (la mujer en la II Republica)', '25/03/2021'): None,
    ('Menudos heroes', '04/12/2015'): None,
    ('Latir. Reto 3.355', '26/03/2021'): None,
    ('Hostal España', '30/09/2020'): None,
    ('Construyendo la luz', '10/12/2021'): None,
    ('Mi profesor', '27/06/2021'): None,  # ej: 778230 = Amor solfeando, El (1930)
    ('Traje, El. Algo mas que una leyenda', '20/11/2021'): None,
    ('Marcos y vida', '01/11/2021'): None,
    ('Tiempos de deseo', '05/03/2021'): None,
    ('Stop', '09/06/2021'): None,  # ej: 74403 = A UN METRO DE TI (2009)
    ('Zalamero', '22/10/2020'): None,
    ('¡Maldicion! He vuelto a cambiar', '30/06/2021'): None,
    ('Coda 77', '21/10/2021'): None,
    ('Frederica Montseny, la mujer que habla', '07/10/2021'): None,
    ('Dernier round', '09/12/2021'): None,
    ('Segundos fuera', '02/12/2021'): None,
    ('Neskatoak', '20/10/2021'): None,
    ('Andromedas', '06/03/2021'): None,
    ('Amanecer', '17/09/2021'): None,  # ej: 22996 = Abierto hasta el amanecer (1995)
    ('Lebrijano, El. Un gitano universal', '01/10/2021'): None,
    ('2 Años, 4 Meses (2 Urte, 4 Hilabete)', '04/09/2020'): None,
    ('Siempre jueves (sempre dijous)', '10/06/2021'): None,
    ('Penitencia', '24/11/2021'): None,  # ej: 9104140 = BARRIOS DE LUZ Y PENITENCIA HUELVA 2005 (2005)
    ('Infravivienda', '26/09/2021'): None,
    ('Una isla en el desierto', '30/09/2021'): None,
    ('Tecido resistente', '28/05/2021'): None,
    ('Rebeldes del NOM', '04/04/2021'): None,
    ('Cartas de Akyab', '26/10/2021'): None,
    ('Rural Cops', '04/06/2021'): None,
    ('Free play', '27/09/2021'): None,
    ('Nosotros no nos mataremos con pistolas', '17/06/2022'): None,
    ('Orquesta terrestre, La', '08/04/2022'): None,
    ('Visitante', '11/02/2022'): None,  # ej: 3215740 = CALLER (EL VISITANTE), THE (None)
    ('Para que sirven las canciones de amor', '14/05/2022'): None,
    ('Sandedro', '21/06/2022'): None,
    ('Rabia kontra la makina', '10/09/2022'): None,
    ('Carpetas azules', '08/11/2022'): None,
    ('Aun', '27/04/2022'): None,  # ej: 787230 = Al son de las guitarras (1952)
    ('Pedra I Oli', '26/07/2022'): None,
    ('Margalida', '27/07/2022'): None,
    ('Cine, registro vivo de nuestra memoria', '12/08/2022'): None,
    ('Sinsombrero, Las', '09/03/2022'): None,
    ('Nanas de la cebolla (Tipularen sehaska kanta)', '03/03/2022'): None,
    ('Ulu, un latido universal', '29/09/2017'): None,
    ('Cocina de los hombres, La', '17/12/2022'): None,
    ('Dorothe na Vila', '28/01/2022'): None,
    ('Todas las mujeres que conozco', '24/11/2018'): None,
    ('Invisible. Te podria pasar a ti', '22/11/2022'): None,
    ('33 años de oscuridad', '23/05/2022'): None,
    ('Asi crecen los enanos', '03/11/2022'): None,
    ('Storm', '15/07/2022'): None,  # ej: 186219 = A Stormy Night (2020)
    ('Reflejos mortales', '16/12/2022'): None,
    ('Hijos de las Nubes, la Ultima Colonia, Los', '18/05/2012'): None,
    ('Una de percebes en el Hurtado', '16/06/2022'): None,
    ('Viento que nos mueve, El', '29/09/2022'): None,
    ('Espejo, El', '22/10/2022'): None,  # ej: 112111 = "EL BRUJO" FRENTE AL ESPEJO (2013)
    ('Salvar Tenerife', '09/06/2022'): None,
    ('Nightclubbing: The Birth of Punk Rock in NYC', '29/10/2022'): None,
    ('Elegido, El', '02/09/2016'): None,  # ej: 140606 = ELEGIDO (STONE COUNCIL), EL (2007)
    ('Flowers pop art', '27/03/2022'): None,
    ('Raphaelismo', '28/06/2022'): None,
    ('Enterrar y callar', '14/07/2022'): None,
    ('Barcelona Surf Destination', '31/05/2022'): None,
    ('No se apaguen las estrellas', '17/06/2021'): None,
    ('Exploradores verticales', '20/10/2022'): None,
    ('Kaiser de la Atlantida, El', '10/05/2022'): None,
    ('Arenas de silencio (Sands of silence)', '14/11/2018'): None,
    ('Hotel Colon (confinamiento incluido)', '05/03/2022'): None,
    ('Cenerentola, La -Teatro Real (Opera)', '23/11/2023'): None,
    ('Efecto All 1, El', '02/02/2023'): None,
    ('Yerma 2030: La España VaciLada', '16/11/2023'): None,
    ('Nacer por nacer', '30/06/2023'): None,
    ('In Albis', '28/09/2023'): None,
    ('Hada de las estaciones, El', '27/03/2023'): None,
    ('Tres muertes de Teofilo del Valle, Las', '09/05/2023'): None,
    ('Alegres tiempos "el concierto"', '31/03/2023'): None,
    ('Bienvenido Mr. Banksy', '03/03/2023'): None,
    ('Sinfonia por un nuevo mundo', '18/01/2023'): None,
    ('Crossroads, el viaje circular de Boa Mistura', '18/10/2023'): None,
    ('Ilegales 82', '02/06/2023'): None,
    ('ADN de la memoria, El', '18/12/2023'): None,
    ('Mi barrio: Vallecas', '30/09/2023'): None,
    ('Valhalla, cielo de roca', '09/11/2023'): None,
    ('AlterNativas: Construyendo futuros posibles', '25/04/2023'): None,
    ('Operacion Brooklyn', '25/07/2023'): None,
    ('Naturaleza muerta Sexy Sadie', '27/07/2023'): None,
    ('De eso no hablamos', '13/06/2023'): None,
    ('Naturaleza de lo extraordinario', '12/05/2023'): None,
    ('Maria y la pelicula olvidada', '28/07/2023'): None,
    ('Retablo de las maravillas. Apuntes para una pelicula sobre el Quijote, El', '22/04/2023'): None,
    ('Ignifugas', '29/05/2023'): None,
    ('Caja vacia, La', '28/02/2023'): None,
    ('Cantuña', '22/06/2023'): None,
    ('FREE, Salir es posible', '21/10/2023'): None,
    ('Guillermo Perez Villalta. Del agua y el Mediterraneo', '21/11/2023'): None,
    ('Sender Barayon. Viaje hacia la luz', '18/04/2023'): None,
    ('Memorias rotas', '23/10/2023'): None,  # ej: 222410 = MEMORIAS ROTAS (LA BALADA DEL COMANDANTE MORENO) (2010)
    ('Jose Luis Lopez Vazquez: ¡Que disparate!', '03/06/2023'): None,
    ('Balada perdida, La', '15/09/2023'): None,
    ('Cambio de sentido', '02/11/2023'): None,  # ej: 53407 = CAMBIO DE SENTIDO (2009)
    ('Salvador Segui: Historia de un anarcosindicalista', '20/04/2023'): None,
    ('Todos contra mí', '12/09/2023'): None,
    ('Du Vin dans les Voiles', '23/07/2023'): None,
    ('Hay alguien en el bosque', '10/03/2023'): None,
    ('Flores de tierra quemada. Estrategias del terror en Guatemala y España', '06/05/2023'): None,
    ('Calavera resumida', '29/06/2023'): None,
    ('Evangelio mayor', '28/06/2023'): None,
    ('Casilda. El eco de otros pasos', '02/02/2023'): None,
    ('Donde la vida puede ser', '08/11/2023'): None,
    ('Jota de Saura', '07/10/2016'): None,
    ('Cuando toco un animal', '28/07/2023'): None,
    ('Casting', '07/03/2023'): None,  # ej: 29305 = Casting (2006)
    ('Zarata', '14/11/2023'): None,
    ('Playa de los ahogados, La', '09/10/2015'): None,
    ('Negociadores, Los. Como construir la Paz', '04/06/2023'): None,
    ('Sucro', '11/05/2023'): None,
    ('Abuela y el forastero, La', '13/09/2024'): None,
    ('Una estrella fugaz', '11/10/2024'): None,
    ('Anaga, Naturaleza Infinita', '13/12/2024'): None,
    ('Vida empieza hoy, La', '13/06/2024'): None,  # ej: 13903 = MI VIDA EMPIEZA HOY (2002)
    ('Sergi Mingote K2 Invernal', '07/06/2024'): None,
    ('Salon, El', '21/07/2024'): None,  # ej: 9498640 = ARTHAUS MUSIK: ESA-PEKKA SALONEN (1997)
    ('Dulcinea', '19/06/2024'): None,  # ej: 3997440 = 4.DON QUIJOTE DE LA MANCHA:"DULCINEA DEL TOBOSO"(DIB.ANIMAD (1981)
    ('Escribir no es normal', '18/12/2024'): None,
    ('Irklais per Atlanta', '05/07/2024'): None,
    ('Amor en toda la cara', '07/06/2024'): None,
    ('Ghosts of the Chelsea Hotel (and Other Rock & Roll Stories)', '21/10/2024'): None,
    ('Witch Story, A', '30/09/2024'): None,
    ('Decision de Joaquina, La', '15/12/2024'): None,
    ('Benin, infancia robada', '21/07/2024'): None,
    ('Contra etiqueta, La', '15/04/2024'): None,
    ('Metralla de amor', '20/02/2024'): None,
    ('Jimmy y la tecla magica', '27/07/2024'): None,
    ('Hermandad de los gitanos, La', '19/03/2024'): None,
    ('Pelicula del hierro y la nieve', '14/02/2024'): None,
    ('Experimento Deanie', '24/03/2024'): None,
    ('Diego Vasallo, la posteridad para más tarde', '25/01/2024'): None,
    ('Aroak', '25/09/2024'): None,
    ('Ultimo deseo, El', '05/11/2024'): None,  # ej: 39119 = El cocinero de los últimos deseos (2017)
    ('Fábula del escorpión y la rana', '22/02/2024'): None,
    ('Dobla la esquina, el volcán', '21/02/2024'): None,
    ('Generacion Lagartija', '21/06/2024'): None,
    ('Artefacto 71', '05/09/2024'): None,
    ('We All Play', '09/07/2024'): None,
    ('Genero chico, El', '14/06/2024'): None,
    ('Lligams, fils que cuiden la vida', '07/05/2024'): None,
    ('Nos parecia importante', '28/06/2024'): None,
    ('Micropolis', '09/05/2024'): None,
    ('Maa-yiem, la extraordinaria historia de Jordi Sabater Pi', '22/02/2024'): None,
    ('Primavera, La', '15/10/2024'): None,  # ej: 100917 = 50 PRIMAVERAS (2017)
    ('Tres caminos a Cádiz', '08/02/2024'): None,
    ('Mans del Genoves, Les', '05/05/2024'): None,
    ('Cyborg Generation', '22/11/2024'): None,
    ('Artifacts Assembly', '04/05/2024'): None,
    ('Maldita primavera, La', '29/06/2024'): None,
    ('Cirlot. La mirada de Bronwyn', '09/05/2024'): None,
    ('Chichi y yo', '11/10/2024'): None,
    ('Teselas. El poder transformador del viaje', '09/03/2024'): None,
    ('Peluquero romantico, El', '09/10/2024'): None,
    ('Valencia 22. City of Design', '05/05/2024'): None,
    ('La Que Se Avecina - Evento Especial', '11/11/2025'): None,
    ('Madres', '22/05/2025'): None,  # ej: 545130 = COMO TODAS LAS MADRES (1944)
    ('Adios Cuba', '12/12/2025'): None,
    ('Sekeleka: Ciudad refugio', '29/05/2025'): None,
    ('Impulso nomada, El', '05/09/2025'): None,
    ('Tamaño del corazon, El', '25/02/2025'): None,
    ('Flores bajo el hielo', '12/09/2024'): None,
    ('Silencio', '15/11/2025'): None,  # ej: 9513240 = AL SILENCIO. CRISTINO DE VERA (2005)
    ('¿Donde estabas cuando estabas?', '19/12/2025'): None,
    ('Love U Carme', '20/11/2025'): None,
    ('Piko & Pala', '28/03/2025'): None,
    ('Chhaupadi', '30/01/2025'): None,
    ('Refugios de papel', '29/07/2025'): None,
    ('Montserrat: entre la roca y el conflicto', '18/09/2025'): None,
    ('Pantalones a la luna', '11/02/2025'): None,
    ('Un viaje hacia nosotros', '19/12/2021'): None,
    ('Reset - Capitulo 1 Ulpotha', '21/01/2025'): None,
    ('Compas del silencio', '31/07/2025'): None,
    ('Rotspanier. Los esclavos españoles del nazismo', '13/12/2025'): None,
    ('Constelacion Portabella', '24/10/2025'): None,
    ('A dos velas', '18/02/2025'): None,
    ('Tu me abrasas', '29/06/2025'): None,
    ('Negro tiene nombre, El', '06/06/2025'): None,
    ('Como conquistamos el Oeste y adonde nos llevo', '27/06/2025'): None,
    ('Naufrafados, Os', '05/09/2025'): None,
    ('Camino de Cova Eiros, O', '30/10/2025'): None,
    ('Gran familia, La', '23/12/2025'): None,  # ej: 71612 = GRAN FAMILIA ESPAÑOLA, LA (2013)
    ('En clave de sol', '07/05/2025'): None,
    ('Castillo de Argueso', '18/01/2025'): None,
    ('Cataluña ambigua', '27/02/2025'): None,
    ('Protagonistes', '03/11/2024'): None,
    ('Walter Benjamin, el aura del camino', '23/10/2025'): None,
    ('Cant dels ocells, El Sagrera', '19/12/2008'): None,
    ('Historias del poder y de la vida', '14/01/2025'): None,
    ('Aminetu', '08/11/2025'): None,
    ('TuentifourSeven', '01/12/2025'): None,
    ('Deuses de Pedra', '26/11/2025'): None,
    ('Nubes y claros', '21/06/2025'): None,
    ('Hijos de Dios', '16/10/2025'): None,  # ej: 201687 = Hijos de un dios menor (1986)
    ('Voladura 76', '12/11/2025'): None,
    ('Triptico', '17/01/2025'): None,  # ej: 194918 = El final del tríptico (2018)
    ('Rojo clavel', '20/11/2025'): None,
    ('Al oeste, en Zapata', '26/11/2025'): None,
    ('Sense filTRES', '14/11/2025'): None,
    ('Comercial, El', '26/05/2025'): None,  # ej: 2113640 = CENTAS VIDEO BASIC COMERCIAL (None)
    ('Ruta. Vol 2: Ibiza, La', '24/10/2025'): None,
    ('Llamame Paul', '14/11/2024'): None,
    ('Epilogo', '28/07/2025'): None,  # ej: 175020 = EL PADRINO DE MARIO PUZO, EPÍLOGO: LA MUERTE DE MICHAEL CORLEONE (2020)
    ('117', '12/06/2025'): None,  # ej: 9051340 = CAMPEONES OLIVER Y BENJI EPISODIOS 117-119 (1983)
    ('Llapis horitzo', '24/10/2025'): None,
    ('Nada', '21/05/2025'): None,  # ej: 194624 = ¿CUÁNTO CUESTA LA NADA? (2025)
    ('Quieres salir puedes entrar', '06/09/2025'): None,
    ('Negro liimbo', '07/06/2025'): None,
    ('Orandi', '19/06/2025'): None,
    ('Todos Vosotros sois Capitanes', '03/06/2011'): None,
    ('Niñas de arena', '27/05/2025'): None,
    ('Invasion pequeña', '04/07/2025'): None,
    ('Helios Gomez: Tinta y Municion', '29/05/2025'): None,
    ('Alyonka soñada', '24/07/2025'): None,
    ('Tierra normal', '15/05/2025'): None,
    ('Mirande', '21/05/2025'): None,
    ('Burnout', '15/05/2025'): None,
    ('Ese mundo que no te da nada', '12/03/2025'): None,
    ('Memoria inmortal, La', '20/02/2025'): None,
    ('Infiltrada en el bunker', '23/10/2025'): None,
    ('Faltan vuestros nombres', '10/05/2025'): None,
    ('Que se sepa', '15/11/2024'): None,
    ('Cuando un rio se convierte en mar', '01/11/2025'): None,
    ('Semana Santa (re)', '01/02/2019'): None,  # ej: 8597640 = A CADA PASO. SEMANA SANTA SALAMANCA 2004 (2004)
    ('Post mortem', '01/06/2021'): None,
    ('Que sabemos, Lo', '25/11/2022'): None,
    ('Pere Joan', '26/07/2022'): None,
    ('Tramposos, Los (1959) (re)', '30/05/2022'): None,  # ej: 8233440 = BANDA DE TRAMPOSOS, UNA (2001)
    ('Siete mesas de billar frances (2007) (re)', '20/06/2023'): None,
    ('Ritos sexuales del diablo, Los (1982) (re)', '13/05/2023'): None,
    ('Mundo de Jacques Demy, El (1995) (re)', '27/09/2022'): None,
    ('Silencio antes de Bach, El (2007) (re)', '08/06/2024'): None,
    ('En la linea del horizonte (1993) (re)', '09/04/2024'): None,
    ('Camino, El (1963)', '02/11/2021'): None,  # ej: 53522 = A mitad camino (2022)
    ('091, policia al habla (1960) (re)', '16/05/2025'): None,
    ('No hay nadie', '02/11/2021'): None,  # ej: 43793 = Nadie hablará de nosotras cuando hayamos muerto (1995)
    ('Todo a la vez', '05/09/2021'): None,  # ej: 85922 = Todo a la vez en todas partes (2022)
    ('Feliz no cumpleaños', '07/11/2023'): None,  # ej: 4817040 = POWER RANGERS: FELIZ CUMPLEAÑOS, ZACK; NO HAGAIS EL PAY VOL6 (1993)
    ('Bastarda', '09/05/2024'): None,  # ej: 665740 = CIUDAD LLAMADA BASTARDA, UNA (None)
    ('Desahuciados', '05/02/2025'): None,  # ej: 2316640 = DESAHUCIADOS, LOS (1979)
    ('Antologia de l’atzar', '30/09/2021'): None,
    ("Ultimo tren al rock'n'roll, El", '18/02/2022'): None,
    ("Zoo, Sobreviure a l'incendi", '22/11/2024'): None,
    ("Sexo, drogas, rock 'n' roll y política. Instituto Santamarca, 1975-1985", '15/02/2024'): None,
    ("Dies d'estiu i de pluja", '13/06/2025'): None,
}

In [19]:
OVERRIDES_ICAA_ID.update({k: v for k, v in OVERRIDES_ICAA_ID_NUEVOS.items() if v is not None})

In [20]:
claves_a_aplicar = [k for k, v in OVERRIDES_ICAA_ID_NUEVOS.items() if v is not None]

cache = pd.read_csv(CACHE_ICAA_ID, sep=';')
antes = len(cache)
cache = cache[~cache.apply(lambda f: (f['titulo_original'], f['fecha_estreno_original']) in claves_a_aplicar, axis=1)]
cache.to_csv(CACHE_ICAA_ID, index=False, sep=';')
print(f"Filas eliminadas: {antes - len(cache)}. Filas restantes: {len(cache)}")

Filas eliminadas: 4. Filas restantes: 2206


## Utilidades de mantenimiento (opcional -- correr solo si hace falta)

Estas celdas purgan filas de la caché para forzar su re-procesamiento con la
lógica más reciente. No son parte del flujo normal -- solo tócalas si, tras ver
el resumen de arriba, decides que vale la pena reintentar algún grupo.

In [ ]:
# Purga filas de reposición que sigan sin resolver (incluye sin_coincidencia_exacta_titulo
# Y sin_candidatos_cercanos_probable_reposicion) para reintentarlas con la lógica actual.
import re

cache = pd.read_csv(CACHE_ICAA_ID, sep=';')

patron_reposicion = re.compile(r'\(\s*(?:\d{4}\s*)?(?:4k\s*)?(?:re\s*:?\s*\d{0,4}\s*)?\)|aniversario', re.IGNORECASE)
es_reposicion = cache['titulo_original'].str.contains(patron_reposicion, na=False, regex=True)
pendiente_o_sin_match = cache['icaa_id'].isna() | (cache['estado'] == 'sin_candidatos_cercanos_probable_reposicion')

a_purgar = cache[es_reposicion & pendiente_o_sin_match]
print(f"Filas de reposición a re-intentar: {len(a_purgar)}")

cache = cache[~(es_reposicion & pendiente_o_sin_match)]
cache.to_csv(CACHE_ICAA_ID, index=False, sep=';')
print(f"Filas restantes en caché: {len(cache)}")


In [ ]:
# Purga filas "sin_coincidencia_exacta_titulo" con exactamente 1 candidato --
# casi siempre son casos obsoletos de una versión anterior de la normalización.
import json

cache = pd.read_csv(CACHE_ICAA_ID, sep=';')

def tiene_un_candidato(fila):
    if pd.isna(fila['candidatos_json']):
        return False
    try:
        return len(json.loads(fila['candidatos_json'])) == 1
    except Exception:
        return False

sospechosas = (
    (cache['estado'] == 'sin_coincidencia_exacta_titulo')
    & cache.apply(tiene_un_candidato, axis=1)
)
print(f"Filas sospechosas de estar obsoletas (1 candidato, sin match): {sospechosas.sum()}")

cache = cache[~sospechosas]
cache.to_csv(CACHE_ICAA_ID, index=False, sep=';')
print(f"Filas restantes en caché: {len(cache)}")


### Purgar filas con error de navegación (`icaa_id_busqueda_cache.csv`)

Filas donde el intento de búsqueda falló por timeout/conexión (no por lógica
de matching). Se borran para que el bucle principal las reintente desde cero.

In [33]:
cache = pd.read_csv(CACHE_ICAA_ID, sep=';')
fallidas = cache['estado'] == 'error_navegacion'
print(f"Filas con error de navegación a reintentar: {fallidas.sum()}")

cache = cache[~fallidas]
cache.to_csv(CACHE_ICAA_ID, index=False, sep=';')
print(f"Filas restantes en caché: {len(cache)}")


Filas con error de navegación a reintentar: 0
Filas restantes en caché: 2079


### Purgar fichas fallidas (`ficha_icaa_cache.csv`)

Una ficha que falló por error de red se guarda con `icaa_id` presente pero
todos los demás campos en `None` -- y como la lógica de "ya scrapeado" solo
mira si el `icaa_id` está en caché, esas filas nunca se reintentarían solas.
`subvenciones_total` es el único campo que SIEMPRE se rellena en un scraping
real y exitoso (mínimo `0.0`), así que sirve como indicador fiable de fallo.

In [ ]:
cache_fichas = pd.read_csv(CACHE_FICHAS, sep=';')
fallidas_ficha = cache_fichas['subvenciones_total'].isna()
print(f"Fichas fallidas (sin datos, a reintentar): {fallidas_ficha.sum()}")

cache_fichas = cache_fichas[~fallidas_ficha]
cache_fichas.to_csv(CACHE_FICHAS, index=False, sep=';')
print(f"Filas restantes en caché de fichas: {len(cache_fichas)}")


### Purgar duplicados antes de propagar

In [27]:
resultado_busqueda = pd.read_csv(CACHE_ICAA_ID, sep=';')
antes = len(resultado_busqueda)

resultado_busqueda['_tiene_id'] = resultado_busqueda['icaa_id'].notna()
resultado_busqueda = resultado_busqueda.sort_values('_tiene_id')  # sin id primero, con id al final
resultado_busqueda = resultado_busqueda.drop_duplicates(
    subset=['titulo_original', 'fecha_estreno_original'], keep='last'
).drop(columns='_tiene_id')

print(f"Duplicados eliminados: {antes - len(resultado_busqueda)}")

# Reescribe la caché ya limpia, para que esto no vuelva a pasar en la próxima corrida
resultado_busqueda.to_csv(CACHE_ICAA_ID, index=False, sep=';')

Duplicados eliminados: 131


## Propagar `icaa_id` a todas las filas de `icaa_raw`

In [49]:
icaa_raw = icaa_raw.merge(
    resultado_busqueda[['titulo_original', 'fecha_estreno_original', 'icaa_id']],
    left_on=['titulo', 'fecha_estreno'],
    right_on=['titulo_original', 'fecha_estreno_original'],
    how='left',
    validate='many_to_one',
).drop(columns=['titulo_original', 'fecha_estreno_original'])

print(f"icaa_raw: {len(icaa_raw)} filas, {icaa_raw['icaa_id'].notna().sum()} con icaa_id")
icaa_raw.head()


icaa_raw: 2812 filas, 2504 con icaa_id


,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio,anio_reposicion,titulo_busqueda,articulo,icaa_id
0,A todo tren. Destino Asturias,Warner Bros,08/07/2021,8493358.0,1500811,2021,NaN,A todo tren. Destino Asturias,NaN,157420.0
1,Way Down,Sony,12/11/2021,5628247.0,887897,2021,NaN,Way Down,NaN,41519.0
2,Operacion Camaron,Walt Disney,24/06/2021,3522415.0,597700,2021,NaN,Operacion Camaron,NaN,56519.0
3,"Buen patron, El",Tri Pictures,15/10/2021,3336892.0,528523,2021,NaN,Buen patron,El,159020.0
4,Maixabel,Walt Disney,24/09/2021,2828416.0,515293,2021,NaN,Maixabel,NaN,157320.0


## Scraping de ficha para cada `icaa_id` único resuelto

Distribuidora, género/tipo/tags, nacionalidad, subvenciones, director(es) y
guionista(s) -- una sola petición por `icaa_id` único, con caché reanudable.

In [29]:
icaa_ids_unicos = sorted(icaa_raw['icaa_id'].dropna().unique().astype(int).tolist())
print(f"icaa_id únicos a scrapear: {len(icaa_ids_unicos)}")

ya_scrapeados = set()
if CACHE_FICHAS.exists():
    cache_fichas_previo = pd.read_csv(CACHE_FICHAS, sep=';')
    ya_scrapeados = set(cache_fichas_previo['icaa_id'])
    print(f"Reanudando: {len(ya_scrapeados)} fichas ya scrapeadas")

pendientes_ficha = [i for i in icaa_ids_unicos if i not in ya_scrapeados]
print(f"Pendientes: {len(pendientes_ficha)} / {len(icaa_ids_unicos)}")


icaa_id únicos a scrapear: 1775
Pendientes: 1775 / 1775


In [30]:
driver = iniciar_driver()
write_header = not CACHE_FICHAS.exists()
fallos_consecutivos = 0
MAX_FALLOS_ANTES_DE_REINICIAR = 3

try:
    with open(CACHE_FICHAS, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS_FICHA, delimiter=';', quoting=csv.QUOTE_NONNUMERIC)
        if write_header:
            writer.writeheader()

        for i, icaa_id in enumerate(pendientes_ficha, 1):
            if i % 50 == 0:
                print(f"[{i}/{len(pendientes_ficha)}] icaa_id={icaa_id}")
            try:
                soup = obtener_ficha(icaa_id, driver)
                datos = parsear_ficha_icaa(soup, icaa_id)
                for campo_lista in ("tags_icaa", "directores_icaa", "guionistas_icaa", "subvenciones_icaa", "empresas_productoras_icaa"):
                    datos[campo_lista] = json.dumps(datos[campo_lista], ensure_ascii=False) if datos[campo_lista] else None
                fallos_consecutivos = 0
            except Exception as e:
                print(f"  Error en ficha {icaa_id}: {e}")
                datos = {campo: None for campo in CAMPOS_FICHA}
                datos["icaa_id"] = icaa_id
                fallos_consecutivos += 1
                if fallos_consecutivos >= MAX_FALLOS_ANTES_DE_REINICIAR:
                    print(f"  ⚠ {fallos_consecutivos} fallos seguidos -- reiniciando el driver de Firefox")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    time.sleep(5)
                    driver = iniciar_driver()
                    fallos_consecutivos = 0

            writer.writerow(datos)
            f.flush()
            time.sleep(random.uniform(2.0, 4.0))
finally:
    driver.quit()
    print("✓ Driver cerrado")

print(f"\n✓ Completado -> {CACHE_FICHAS}")


[50/1775] icaa_id=9120
[100/1775] icaa_id=16917
[150/1775] icaa_id=23120
[200/1775] icaa_id=30020
[250/1775] icaa_id=36022
[300/1775] icaa_id=43216
[350/1775] icaa_id=49116
[400/1775] icaa_id=53623
[450/1775] icaa_id=58420
[500/1775] icaa_id=62725
[550/1775] icaa_id=69119
[600/1775] icaa_id=78215
[650/1775] icaa_id=83519
[700/1775] icaa_id=88119
[750/1775] icaa_id=92920
[800/1775] icaa_id=98219
[850/1775] icaa_id=103120
[900/1775] icaa_id=108422
[950/1775] icaa_id=116121
[1000/1775] icaa_id=124422
[1050/1775] icaa_id=132918
[1100/1775] icaa_id=140423
[1150/1775] icaa_id=146319
[1200/1775] icaa_id=152421
[1250/1775] icaa_id=157510
[1300/1775] icaa_id=159920
[1350/1775] icaa_id=167821
[1400/1775] icaa_id=177417
[1450/1775] icaa_id=183223
[1500/1775] icaa_id=190923
[1550/1775] icaa_id=197219
[1600/1775] icaa_id=210250
[1650/1775] icaa_id=225025
[1700/1775] icaa_id=249523
[1750/1775] icaa_id=545051
✓ Driver cerrado

✓ Completado -> ..\3 - csv\ficha_icaa_cache.csv


In [ ]:
fichas = pd.read_csv(CACHE_FICHAS, sep=';')

## Overrides manuales de distribuidora

In [31]:
fichas = pd.read_csv(CACHE_FICHAS, sep=';')
icaa_raw_temp = icaa_raw.merge(fichas, on='icaa_id', how='left', validate='many_to_one')

sin_distribuidora = icaa_raw_temp[
    icaa_raw_temp['distribuidora_nacional_icaa'].isna()
    & icaa_raw_temp['distribuidora_pdf_heuristica'].isna()
]

print(f"Filas sin ninguna distribuidora (ni ficha ni heurística): {len(sin_distribuidora)}")
sin_distribuidora[['titulo', 'fecha_estreno', 'icaa_id']].drop_duplicates()

Filas sin ninguna distribuidora (ni ficha ni heurística): 1


,titulo,fecha_estreno,icaa_id
2539,"Cant dels ocells, El Sagrera",19/12/2008,NaN


In [32]:
OVERRIDES_DISTRIBUIDORA = {
    # ('TITULO', '23/11/2023'): 'Proyecfilm',
    ('Cant dels ocells, El Sagrera', '19/12/2008'): 'Sagrera',

}
print(f"✓ {len(OVERRIDES_DISTRIBUIDORA)} overrides de distribuidora cargados")


✓ 1 overrides de distribuidora cargados


## Agrupación final -> `icaa_peliculas`

Se agrupa por `icaa_id` cuando está resuelto; para las filas sin `icaa_id` se sigue
agrupando por `(titulo, fecha_estreno)` como respaldo. `subvenciones_total` hereda
automáticamente la convención ya establecida: `0.0` cuando `icaa_id` existe y la
ficha se scrapeó (ausencia confirmada), `NaN` cuando no hay `icaa_id` (sin verificar) --
no hace falta código extra, sale solo del `merge`.

In [50]:
fichas = pd.read_csv(CACHE_FICHAS, sep=';')
icaa_raw = icaa_raw.merge(fichas, on='icaa_id', how='left', validate='many_to_one')

icaa_raw['clave_grupo'] = icaa_raw['icaa_id'].astype('Int64').astype(str)
icaa_raw.loc[icaa_raw['icaa_id'].isna(), 'clave_grupo'] = (
    icaa_raw.loc[icaa_raw['icaa_id'].isna(), 'titulo'] + '||' +
    icaa_raw.loc[icaa_raw['icaa_id'].isna(), 'fecha_estreno']
)

def _agrupar(grupo):
    fila = grupo.iloc[0].copy()
    fila['recaudacion_total'] = grupo['recaudacion'].sum()
    fila['espectadores_total'] = grupo['espectadores'].sum()
    anios = sorted(set(grupo['anio'].astype(str)))
    fila['anios_en_cartelera'] = ','.join(anios)
    fila['num_anios'] = len(anios)

    clave = (fila['titulo'], fila['fecha_estreno'])
    dist_nacional = fila.get('distribuidora_nacional_icaa')
    dist_pdf = fila.get('distribuidora_pdf_heuristica')

    if clave in OVERRIDES_DISTRIBUIDORA:
        fila['distribuidora_final'] = OVERRIDES_DISTRIBUIDORA[clave]
        fila['distribuidora_fuente'] = 'override_manual'
    elif pd.notna(dist_nacional) and dist_nacional:
        fila['distribuidora_final'] = dist_nacional
        fila['distribuidora_fuente'] = 'ficha_icaa'
    else:
        fila['distribuidora_final'] = dist_pdf
        fila['distribuidora_fuente'] = 'pdf_heuristica' if pd.notna(dist_pdf) else None

    return fila

icaa_peliculas = icaa_raw.groupby('clave_grupo', group_keys=False).apply(_agrupar).reset_index(drop=True)
#icaa_peliculas = icaa_peliculas.drop(columns=['clave_grupo'])
icaa_peliculas = icaa_peliculas.drop(columns='clave_grupo', errors='ignore')
print(f"icaa_peliculas: {len(icaa_peliculas)} filas (de {len(icaa_raw)} en icaa_raw)")
con_subvencion = (icaa_peliculas['subvenciones_total'] > 0).sum()
print(f"Con subvención > 0: {con_subvencion}")
icaa_peliculas.head()


icaa_peliculas: 2052 filas (de 2812 en icaa_raw)
Con subvención > 0: 590


,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio,anio_reposicion,titulo_busqueda,articulo,icaa_id,...,guionistas_icaa,subvenciones_icaa,subvenciones_total,empresas_productoras_icaa,recaudacion_total,espectadores_total,anios_en_cartelera,num_anios,distribuidora_final,distribuidora_fuente
0,"091, policia al habla (1960) (re)",Independent,16/05/2025,42.0,12,2025,1960.0,"091, policia al habla",NaN,NaN,...,NaN,NaN,NaN,NaN,42.0,12,2025,1,Independent,pdf_heuristica
1,Todos lo saben,Universal,14/09/2018,83.0,18,2022,NaN,Todos lo saben,NaN,100317.0,...,"[""Asghar Farhadi""]","[{""concepto"": ""Ayudas Generales para la produc...",280000.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 50.0, ""...",83.0,18,2022,1,"UNIVERSAL PICTURES INTERNATIONAL SPAIN, S.L. (...",ficha_icaa
2,"Generacion silenciosa, La",Independent,10/09/2021,129.0,35,2021,NaN,Generacion silenciosa,La,100421.0,...,"[""Ferrán Navarro-Beltrán""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",129.0,35,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
3,Platon,Independent,08/10/2021,237.0,60,2021,NaN,Platon,NaN,100517.0,...,"[""Iván López González""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",237.0,60,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
4,Agua,Independent,11/02/2022,1975.0,319,2022,NaN,Agua,NaN,100820.0,...,"[""Vicente Pérez Herrero"", ""Marta Martínez Rodr...",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",2296.0,357,"2022,2023",2,LAS PRODUCCIONES DE LOS IMAGINARIOS. PRODUCCIO...,ficha_icaa


In [51]:
icaa_peliculas.head(5)

,titulo,distribuidora_pdf_heuristica,fecha_estreno,recaudacion,espectadores,anio,anio_reposicion,titulo_busqueda,articulo,icaa_id,...,guionistas_icaa,subvenciones_icaa,subvenciones_total,empresas_productoras_icaa,recaudacion_total,espectadores_total,anios_en_cartelera,num_anios,distribuidora_final,distribuidora_fuente
0,"091, policia al habla (1960) (re)",Independent,16/05/2025,42.0,12,2025,1960.0,"091, policia al habla",NaN,NaN,...,NaN,NaN,NaN,NaN,42.0,12,2025,1,Independent,pdf_heuristica
1,Todos lo saben,Universal,14/09/2018,83.0,18,2022,NaN,Todos lo saben,NaN,100317.0,...,"[""Asghar Farhadi""]","[{""concepto"": ""Ayudas Generales para la produc...",280000.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 50.0, ""...",83.0,18,2022,1,"UNIVERSAL PICTURES INTERNATIONAL SPAIN, S.L. (...",ficha_icaa
2,"Generacion silenciosa, La",Independent,10/09/2021,129.0,35,2021,NaN,Generacion silenciosa,La,100421.0,...,"[""Ferrán Navarro-Beltrán""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",129.0,35,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
3,Platon,Independent,08/10/2021,237.0,60,2021,NaN,Platon,NaN,100517.0,...,"[""Iván López González""]",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",237.0,60,2021,1,PELICULA SIN DISTRIBUIDORA ASIGNADA (98J000),ficha_icaa
4,Agua,Independent,11/02/2022,1975.0,319,2022,NaN,Agua,NaN,100820.0,...,"[""Vicente Pérez Herrero"", ""Marta Martínez Rodr...",NaN,0.0,"[{""pais"": ""ESPAÑA"", ""porcentaje_pais"": 100.0, ...",2296.0,357,"2022,2023",2,LAS PRODUCCIONES DE LOS IMAGINARIOS. PRODUCCIO...,ficha_icaa


In [52]:
mascara = (
    (icaa_peliculas['titulo'] == 'Cant dels ocells, El Sagrera')
    & (icaa_peliculas['fecha_estreno'] == '19/12/2008')
)

print("Filas encontradas:", mascara.sum())
print(icaa_peliculas.loc[mascara, ['titulo', 'fecha_estreno', 'distribuidora_final', 'distribuidora_fuente']])

icaa_peliculas.loc[mascara, 'distribuidora_final'] = 'Sagrera'
icaa_peliculas.loc[mascara, 'distribuidora_fuente'] = 'override_manual'

print("\nDespués del cambio:")
print(icaa_peliculas.loc[mascara, ['titulo', 'fecha_estreno', 'distribuidora_final', 'distribuidora_fuente']])

Filas encontradas: 1
                            titulo fecha_estreno distribuidora_final  \
1814  Cant dels ocells, El Sagrera    19/12/2008             Sagrera   

     distribuidora_fuente  
1814      override_manual  

Después del cambio:
                            titulo fecha_estreno distribuidora_final  \
1814  Cant dels ocells, El Sagrera    19/12/2008             Sagrera   

     distribuidora_fuente  
1814      override_manual  


In [53]:
mascara = (
    (icaa_peliculas['titulo'] == 'Cant dels ocells, El Sagrera')
    & (icaa_peliculas['fecha_estreno'] == '19/12/2008')
)

print("Filas encontradas:", mascara.sum())

icaa_peliculas.loc[mascara, 'titulo'] = 'Cant dels ocells, El'
icaa_peliculas.loc[mascara, 'distribuidora_final'] = 'Sagrera'
icaa_peliculas.loc[mascara, 'distribuidora_fuente'] = 'override_manual'
icaa_peliculas.loc[mascara, 'titulo_busqueda'] = 'Cant dels ocells'
icaa_peliculas.loc[mascara, 'articulo'] = 'El'

print(icaa_peliculas.loc[mascara, ['titulo', 'fecha_estreno', 'distribuidora_final', 'distribuidora_fuente']])

Filas encontradas: 1
                    titulo fecha_estreno distribuidora_final  \
1814  Cant dels ocells, El    19/12/2008             Sagrera   

     distribuidora_fuente  
1814      override_manual  


## Guardar en MySQL y CSV

In [54]:
icaa_raw.to_sql('icaa_raw', engine, if_exists='replace', index=False)
icaa_peliculas.to_sql('icaa_peliculas', engine, if_exists='replace', index=False)
print("✓ Tablas actualizadas en MySQL")

engine.dispose()
print("MySQL conexión cerrada.")

icaa_raw.to_csv(BASE / '3 - csv' / 'icaa_raw.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
icaa_peliculas.to_csv(BASE / '3 - csv' / 'icaa_peliculas.csv', index=False, sep=';', quoting=csv.QUOTE_NONNUMERIC)
print("✓ CSVs exportados")


✓ Tablas actualizadas en MySQL
MySQL conexión cerrada.
✓ CSVs exportados
